> 🚨 **Folders required before running this Notebook: Building Block `.xyz` Files**

- `NODE_FILES` *(.xyz, required)*  
  Files containing the node fragments / higher-connectivity building units.  
  **Location:** `0_node/` directory  
  **Default behavior:** The required number of node files depends on the selected `TOPO`:  
  - `hcb` / `sql` / `kgm`: exactly one `.xyz` file must be present  
  - `hcb_ab`: exactly two `.xyz` files must be present  
  Otherwise, provide explicit paths via `input_nodes=[...]`.

- `LINKER_FILES` *(.xyz, topology-dependent)*  
  Files containing the linker fragments / lower-connectivity building units.  
  **Location:** `0_linker/` directory  
  **Default behavior:** The required number of linker files depends on the selected `TOPO`:  
  - `hcb` / `sql` / `kgm`: exactly one `.xyz` file must be present  
  - `hcb_ab`: no linker file is required; files in `0_linker/` are ignored  
  Pass `input_linkers=[]` to explicitly provide no linkers, or provide explicit paths via `input_linkers=[...]`.

> 🚨 **Building Block Requirements**
>
> - **Dummy atoms are required at every intended connection site.**  
>   Each position at which a bond will be formed in the final COF must be marked by a dummy atom in the input building block.
>
> - **Dummy atom encoding**
>   - `He` = connection site (all bonds are treated as single)
>
> - **Important**  
>   Dummy atoms are used only to define the connection sites during construction. They do not represent real atoms in the final COF structure.


In [ ]:
import coflandscaper as cl

### General Settings & Options

#### Structural Parameters

- `TOPOLOGY` *(str, required, default: None)*  
  Defines the underlying network topology.  
  **Allowed values:** `hcb`, `sql`, `hcb_ab`, `kgm`  
  - `hcb` / `sql` / `kgm`: require one node file and one linker file  
  - `hcb_ab`: requires two node files and no linker file

#### Naming Parameters

- `COF_NAME` *(str, required, default: None)*  
  Unique identifier for the generated structure.  
  All output files and directories will be named using this value.  
  **Constraints:**  
  - Should be unique within the working directory.  
  - Avoid spaces and special characters for compatibility.

#### Stacking Parameters

- `MODE` *(str, required, default: None)*  
  Defines the stacking mode(s) to be generated.  
  **Allowed values:**  
  - `incl` → inclined stacking  
  - `serr` → serrated stacking  
  - `both` → generate both stacking modes

  **Behavior:**  
  Selecting `both` will generate both configurations sequentially and store them in the corresponding output structure.


In [ ]:
TOPOLOGY = "sql"
COF_NAME = "COF-1"
MODE = "both"

### Single-Layer COF Construction & Pre-Optimization

This step generates the single-layer COF structure and performs MACE pre-optimization.

> **Practical note:** If the input building-block fragments are strongly non-planar, the generated stacking matrix can contain strained or sterically congested geometries. Starting from relatively planar building units is therefore recommended, followed by relaxation of the assembled framework. Because the isolated single-layer model has more conformational freedom than the final stacked COF, the preoptimized layer may twist more strongly than the corresponding bulk structure. The generated matrix should be inspected for unphysical close contacts before downstream screening.

In [ ]:
builder = cl.BuildCOF2D()
builder.build(topo=TOPOLOGY, cof_name=COF_NAME)

In [ ]:
preopt = cl.MaceOpt()
preopt.run_preopt(cof_name=COF_NAME)

### ILD × ILS Structure Matrix Generation

This step generates stacked COF structures by systematically varying:

- **Interlayer Distance (ILD)** (default range: 3.0-4.0 A in steps of 0.1 A)  
- **Interlayer Slipping (ILS)** (default range: 0.0 A to the AB limit in steps of 1.0 A)  
  Default ILS shift length and angle are auto-computed from the topology; use `print_shift=True` to display them.

The input structure is the pre-optimized single-layer (AA stacking with large interlayer distance).  
This step converts it into physically meaningful bulk stacking configurations.

Generated stacked structures organized by mode:  
- `.../serr/` → serrated stacking configurations  
- `.../incl/` → inclined stacking configurations  

These structures are intended for subsequent single-point energy evaluations and energy landscape analysis.


In [ ]:
matrix = cl.CreateMatrix()
matrix.run(cof_name=COF_NAME, topo=TOPOLOGY, mode=MODE)

### MACE Single-Point Energy Evaluation

This step computes single-point energies for generated structures using `MaceSP` (no geometry optimization).

Single-point energies are written ro `{COF_NAME}_sp_energies_{serr|incl}.csv` in `COF_NAME/3_{COF_NAME}_landscape/`  

In [ ]:
sp = cl.MaceSP()
sp.run_mode(cof_name=COF_NAME, mode=MODE)

### Potential Energy Landscape (PES)

This step plots an approximate stacking potential energy surface by mapping the relative energies of the generated ILD/ILS structure matrix as a function of interlayer distance (ILD) and interlayer slipping (ILS).

Because no full structural relaxation is performed at this stage, the PES is treated as a reduced-dimensional screening model rather than a full description of the underlying high-dimensional energy landscape. Within this approximation, it provides a qualitative to semi-quantitative representation of the relative stability of different stacking arrangements and serves to identify candidate minima for subsequent refinement by full geometry optimization.

---

- `show` *(bool, optional, default: `False`)*  
  If `True`, displays interactive plot windows.  

- PES plots `pes_{COF_NAME}_{serr|incl}_{plot_mode}.png` default written to:  
  `COF_NAME/3_{COF_NAME}_landscape/`

In [ ]:
landscape = cl.Landscape()
landscape.run_mode(cof_name=COF_NAME, mode=MODE, show=True)

### Structure Selection for Optimization

This step selects candidate structures corresponding to automatically detected local or gloabl minima are copies them into dedicated folders.

In [ ]:
selector = cl.SelectCofs()
selector.run_mode(cof_name=COF_NAME, mode=MODE)

### MACE Geometry Optimization

This step performs geometry optimizations of selected stacking structures using `MaceOpt`.

`MaceOpt` uses ASE `FrechetCellFilter` + `LBFGS`, writes optimized CIF files, and can write a combined energy CSV (`{COF_NAME}_opt_energies_per_layer.csv`).

  Optimized structures (`.cif`) and optional energies CSV are written to:  
  `COF_NAME/4_{COF_NAME}_optimization/`

In [ ]:
opt = cl.MaceOpt()
opt.run_mode(cof_name=COF_NAME, mode=MODE)

### Analysis & Visualization

This step analyzes optimized structures by computing interlayer distance (ILD) and interlayer slipping (ILS) from the optimized structures, and writing a summary CSV for comparison across stacking configurations.

In addition, structures can be visualized using an interactive viewer.

In [ ]:
# Defaults
analyzer = cl.AnalyzeStacking()
analyzer.analyze(cof_name=COF_NAME, mode=MODE)

In [ ]:
# Defaults
visualizer = cl.VisualizeCOF()
visualizer.visualize_cof(cof_name=COF_NAME, mode=MODE)

### PXRD Pattern Generation

This step simulates PXRD patterns from optimized CIF structures and writes per-structure `.xy` files.

---

#### Parameters

- `wavelength` *(str, optional, default: `CuKa`)*  
  X-ray source line used for simulation.  
  **Allowed values:** `CuKa`, `MoKa`, `CrKa`, `FeKa`, `CoKa`, `AgKa`  
  Choose this to match your instrument/source.

- `two_theta_range` *(tuple[float, float], optional, default: `(1.5, 60.0)`)*  
  Angular simulation window in degrees for generated `.xy` data.

- PXRD data (`.xy`) default written to:  
  `COF_NAME/5_{COF_NAME}_analysis/pxrd_xy/{serr|incl}/`


In [ ]:
pxrd = cl.PXRD(wavelength="CuKa", two_theta_range=(1.5, 60.0))
pxrd.run(cof_name=COF_NAME, mode=MODE)

### PXRD Peak Extraction

This step extracts the strongest simulated PXRD peaks from generated `.xy` files and writes peak tables.

---

#### Parameters

- `max_peaks` *(int, optional, default: `10`)*  
  Maximum number of peaks reported per structure.

- `min_relative_intensity` *(float, optional, default: `1.0`)*  
  Minimum relative intensity threshold in %. Peaks below this value are ignored.

- Peak tables (`.csv`) default written to:  
  `COF_NAME/5_{COF_NAME}_analysis/pxrd_peaks/{serr|incl}/`

In [ ]:
pxrd.extract_peaks(cof_name=COF_NAME, mode=MODE)

### PXRD Plot

This step provides two plotting helpers:

- `plot_sim`  
  Stacked simulated PXRD patterns from `pxrd_xy/{serr|incl}` (or `pxrd_xy_dft/{serr|incl}` when `dft=True`).

- `plot_sim_vs_exp`  
  One experimental `.xy` file compared against every simulated `.xy` file in its own row.

#### Parameters

- `exp_xy_file` *(str or Path, optional, default: None)*  
  Path to the experimental `.xy` file.  
  **Default behavior:** If not provided, the folder `experimental_pxrd` (in the current directory) is searched for exactly one `.xy` file. If multiple `.xy` files exist, you must specify the exact path explicitly.  

- `xy_folder` for `plot_sim` or `simulated_xy_folder` for `plot_sim_vs_exp` *(str or Path, optional, default: None)*  
  Path to the simulated `.xy` files in their `MODE` subfolders.  
  **Custom labels:** To change the label displayed in the plot, simply rename your `.xy` file before passing it.

- `xlim` *(tuple[float, float], optional, default: `(1.5, 60.0)`)*  
  X-axis bounds as minimum and maximum 2θ range [º].

#### Output

- `plot_sim` writes `{COF_NAME}_sim_{serr|incl}.png`.  
- `plot_sim_vs_exp` writes `{COF_NAME}_{serr|incl|both}.png`.

In [ ]:
pxrd.plot_sim(
    cof_name=COF_NAME,
    mode=MODE,
    xy_folder=None,
    xlim=(1.5, 60.0),
)

In [ ]:
# For an example of `PXRD.plot_sim_vs_exp()`, please refer to the hybrid workflow example. Experimental `.xy` data are not available for COF-1, so this functionality cannot be demonstrated here.